In [0]:
from pyspark.sql.functions import col, hour, to_date, count, sum, avg, round

silver_table_name = "nyc_taxi_etl.default.silver_yellow_tripdata"
gold_table_name = "nyc_taxi_etl.default.gold_hourly_metrics"
gold_table_path = "abfss://bronzedata@storagenyctaxietl.dfs.core.windows.net/gold/hourly_metrics"

try:
    print("1. Reading clean Silver data...")
    silver_df = spark.read.table(silver_table_name)
    
    print("2. Aggregating business metrics for the Gold Layer...")
    gold_df = silver_df.withColumn("pickup_date", to_date(col("tpep_pickup_datetime"))) \
                       .withColumn("pickup_hour", hour(col("tpep_pickup_datetime")))
    
    gold_df = gold_df.groupBy("pickup_date", "pickup_hour").agg(
        count("*").alias("total_trips"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("total_amount"), 2).alias("avg_fare_per_trip"),
        round(avg("trip_distance"), 2).alias("avg_distance_miles"),
        round(avg("trip_duration_minutes"), 2).alias("avg_duration_minutes")
    )
    
    print("3. Saving to Gold External Delta Table...")
    gold_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("path", gold_table_path) \
        .saveAsTable(gold_table_name)
        
    print("Success! Gold layer table created and ready for analysis.")

except Exception as e:
    print(f"Gold layer processing failed: {e}")

1. Reading clean Silver data...
2. Aggregating business metrics for the Gold Layer...
3. Saving to Gold External Delta Table...
Success! Gold layer table created and ready for analysis.
